In [3]:
# Add the parent directory of the current working directory to the Python path at runtime. 
# In order to import modules from the src directory.
import os
import sys 

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

In [4]:
import numpy as np
from src.anl_utils import weighted_jaccard

In [5]:
def compute_loo_fits(individual_bats):
    """
    Compute the correlation coefficient and weighted Jaccard index
    between each individual bat's Arnold tongue and the average of the
    remaining bats' Arnold tongues (leave-one-out approach).

    Parameters
    ----------
    individual_bats : array_like
        The individual behavioural Arnold tongues.

    Returns
    -------
    corrcoefs : array_like
        The correlation coefficient
    jaccards : array_like
        The weighted Jaccard index
    """

    num_subjects = individual_bats.shape[0]
    corrcoefs = np.zeros(num_subjects)
    jaccards = np.zeros(num_subjects)

    for i in range(num_subjects):
        left_out_bat = individual_bats[i].flatten()

        average_bat = np.delete(individual_bats, i,
                                axis=0).mean(axis=0).flatten()
        
        min_train = average_bat.min()
        max_train = average_bat.max()
        average_bat = (average_bat - min_train) / (max_train - min_train)
        left_out_bat = (left_out_bat - min_train) / (max_train - min_train)

        corrcoefs[i] = np.corrcoef(left_out_bat, average_bat)[0, 1]
        jaccards[i] = weighted_jaccard(left_out_bat, average_bat)

    
    return corrcoefs, jaccards



In [6]:
empirical_bats_file = '../results/empirical/session_1/individual_bats.npy'
learning_sims_file = '../results/simulation/learning_simulation.npz'
exploration_sims_file = '../results/simulation/parameter_space_exploration.npz'

with np.load(learning_sims_file) as model_fits:
        model_correlation = model_fits['correlation_fits'][:,0]
        model_jaccard = model_fits['jaccard_fits'][:,0]

with np.load(exploration_sims_file) as model_fits:
        range_correlation = model_fits['correlation_fits']
        range_jaccard = model_fits['jaccard_fits']


empirical_bats = np.load(empirical_bats_file)
empirical_correlation, empirical_jaccard = compute_loo_fits(empirical_bats)


proximity to ceiling

In [15]:
x = model_correlation / empirical_correlation
print(f'per subject: {x}')
print(f'mean: {x.mean()}')

per subject: [0.99364123 0.88268903 0.7739333  1.02215062 0.79690979 0.97331769
 0.89800562 0.81547954]
mean: 0.8945158522116248


In [ ]:
x = model_jaccard / empirical_jaccard
print(f'per subject: {x}')
print(f'mean: {x.mean()}')

[0.64708795 0.77007234 0.6948683  1.55063283 0.62137644 0.71339175
 0.61981232 0.69341404]
0.7888319964638537


Improvement over misspecified model (worst parameter settings)

In [ ]:
x = model_correlation / range_correlation.min()
print(f'per subject: {x}')
print(f'mean: {x.mean()}')

[1.71936011 1.44858489 1.54451506 0.36855809 1.49243457 1.75988966
 1.64339633 1.26374352]
1.4050602782607484


In [16]:
x = model_jaccard / range_jaccard.min()
print(f'per subject: {x}')
print(f'mean: {x.mean()}')

per subject: [3.13936369 3.33493704 2.72216207 2.44408691 2.9529489  3.66178258
 2.89160795 2.99945043]
mean: 3.018292446867635
